In [1]:
!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    peft \
    trl \
    datasets \
    wandb

In [2]:
from huggingface_hub import login
import wandb

login()

wandb.init(
    project="Order_of_Operations",
    name="pipeline_b_qlora",
    id="pipeline_b",
    resume="allow",
    config={
        "model": "google/gemma-2-2b",
        "quantization": "NF4_4bit",
        "lora_r": 16,
        "lora_alpha": 32,
        "dataset": "alpaca_500",
        "pipeline": "B_QLoRA"
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: sagarsharma-ai (sagarsharma-ai-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-2-2b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, # compute in bf16, store in 4-bit
    bnb_4bit_use_double_quant=True        # quantize the quantization constants too
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # required for SFT with causal models

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [4]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# expect: ~1-2% of total params are trainable

trainable params: 6,389,760 || all params: 2,620,731,648 || trainable%: 0.2438


In [5]:
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca", split="train")

# Take 800 samples — enough to teach instruction following, not enough to overfit
dataset = dataset.shuffle(seed=42).select(range(800))

def format_alpaca(example):
    if example["input"]:
        prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    else:
        prompt = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    return {"text": prompt}

dataset = dataset.map(format_alpaca, remove_columns=dataset.column_names)
print(dataset[0]["text"])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

### Instruction:
What would be the best type of exercise for a person who has arthritis?

### Response:
For someone with arthritis, the best type of exercise would be low-impact activities like yoga, swimming, or walking. These exercises provide the benefits of exercise without exacerbating the symptoms of arthritis.


In [6]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./gemma2b_qlora_checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="wandb",
    run_name="pipeline_b_qlora",
    seed=42,
    packing=False
)

def formatting_func(example):
    text = example["text"]
    tokens = tokenizer(text, truncation=True, max_length=512)
    return tokenizer.decode(tokens["input_ids"], skip_special_tokens=True)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    formatting_func=formatting_func,
)

Applying formatting function to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

In [7]:
trainer.train()

Step,Training Loss
10,1.998067
20,1.460434
30,1.364154
40,1.411063
50,1.310789
60,1.266427
70,1.243269
80,1.248798
90,1.211111
100,1.300809


TrainOutput(global_step=100, training_loss=1.3814920330047606, metrics={'train_runtime': 1722.7072, 'train_samples_per_second': 0.929, 'train_steps_per_second': 0.058, 'total_flos': 2126993950642176.0, 'train_loss': 1.3814920330047606})

In [8]:
# Save LoRA adapters only (small, ~50MB)
trainer.model.save_pretrained("./pipeline_b_lora_adapters")
tokenizer.save_pretrained("./pipeline_b_lora_adapters")

('./pipeline_b_lora_adapters/tokenizer_config.json',
 './pipeline_b_lora_adapters/tokenizer.json')

In [12]:
wandb.save('./pipeline_b_lora_adapters/tokenizer_config.json')
wandb.save('./pipeline_b_lora_adapters/tokenizer.json')

['/content/wandb/run-20260325_083949-pipeline_b/files/pipeline_b_lora_adapters/tokenizer.json']

In [14]:
from google.colab import drive
drive.mount('/content/drive')


trainer.model.save_pretrained("/content/drive/MyDrive/pipeline_b_lora_adapters")
tokenizer.save_pretrained("/content/drive/MyDrive/pipeline_b_lora_adapters")

Mounted at /content/drive


('/content/drive/MyDrive/pipeline_b_lora_adapters/tokenizer_config.json',
 '/content/drive/MyDrive/pipeline_b_lora_adapters/tokenizer.json')

In [10]:
from peft import PeftModel

# Reload base in 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Attach LoRA adapters
model_b = PeftModel.from_pretrained(base_model, "./pipeline_b_lora_adapters")
model_b.eval()

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
        

In [3]:
import torch
import numpy as np
import wandb
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [4]:
# Kill everything still on GPU
import torch, gc

try:
    del model
    del base_model
    del trainer
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

Free VRAM: 15.53 GB


In [5]:
from google.colab import drive
drive.mount('/content/drive')

model_id = "google/gemma-2-2b"
adapter_path = "/content/drive/MyDrive/pipeline_b_lora_adapters"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
tokenizer.pad_token = tokenizer.eos_token

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

In [6]:
answer_tokens = ["A", "B", "C", "D"]
answer_token_ids = [tokenizer.encode(t, add_special_tokens=False)[0] for t in answer_tokens]
print(dict(zip(answer_tokens, answer_token_ids)))

{'A': 235280, 'B': 235305, 'C': 235288, 'D': 235299}


In [7]:
def get_answer_probs(prompt, model, tokenizer, answer_token_ids):
    """
    Returns softmax probabilities over [A, B, C, D] for a given prompt.
    """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Logits at the last input token position — this is where the model predicts the answer
    last_logits = outputs.logits[0, -1, :]  # shape: (vocab_size,)

    # Extract only A/B/C/D logits
    answer_logits = last_logits[answer_token_ids]  # shape: (4,)

    # Softmax over just these 4 — this is your calibrated probability distribution
    probs = torch.softmax(answer_logits, dim=-1).cpu().numpy()

    return probs  # [P(A), P(B), P(C), P(D)]

In [8]:
def load_mmlu(n=300):
    ds = load_dataset("cais/mmlu", "all", split="test")
    ds = ds.shuffle(seed=42).select(range(n))
    return ds

def load_arc(n=300):
    ds = load_dataset("ai2_arc", "ARC-Challenge", split="test")
    ds = ds.shuffle(seed=42).select(range(n))
    return ds

def load_truthfulqa(n=300):
    ds = load_dataset("truthful_qa", "multiple_choice", split="validation")
    ds = ds.shuffle(seed=42).select(range(n))
    return ds

def format_mmlu(example):
    choices = example["choices"]
    prompt = (
        f"### Instruction:\nAnswer the following question with only the letter A, B, C, or D.\n\n"
        f"Question: {example['question']}\n"
        f"A) {choices[0]}\nB) {choices[1]}\nC) {choices[2]}\nD) {choices[3]}\n\n"
        f"### Response:\n"
    )
    label = example["answer"]  # 0-3 int
    return prompt, label

def format_arc(example):
    labels_map = {"A": 0, "B": 1, "C": 2, "D": 3, "1": 0, "2": 1, "3": 2, "4": 3}
    choices = example["choices"]["text"]
    # ARC sometimes has 3-5 choices, pad to 4 if needed
    while len(choices) < 4:
        choices.append("N/A")
    prompt = (
        f"### Instruction:\nAnswer the following question with only the letter A, B, C, or D.\n\n"
        f"Question: {example['question']}\n"
        f"A) {choices[0]}\nB) {choices[1]}\nC) {choices[2]}\nD) {choices[3]}\n\n"
        f"### Response:\n"
    )
    label = labels_map.get(example["answerKey"], 0)
    return prompt, label

def format_truthfulqa(example):
    choices = example["mc1_targets"]["choices"][:4]
    while len(choices) < 4:
        choices.append("N/A")
    correct_idx = example["mc1_targets"]["labels"].index(1)
    if correct_idx > 3:
        correct_idx = 0
    prompt = (
        f"### Instruction:\nAnswer the following question with only the letter A, B, C, or D.\n\n"
        f"Question: {example['question']}\n"
        f"A) {choices[0]}\nB) {choices[1]}\nC) {choices[2]}\nD) {choices[3]}\n\n"
        f"### Response:\n"
    )
    return prompt, correct_idx

In [10]:
def evaluate_pipeline(model, tokenizer, dataset, format_fn, dataset_name, answer_token_ids):
    results = []

    for i, example in enumerate(dataset):
        try:
            prompt, true_label = format_fn(example)
            probs = get_answer_probs(prompt, model, tokenizer, answer_token_ids)

            pred_label = int(np.argmax(probs))
            confidence = float(probs[pred_label])  # max prob = model's confidence
            is_correct = int(pred_label == true_label)
            true_prob = float(probs[true_label])   # prob assigned to correct answer

            results.append({
                "dataset": dataset_name,
                "true_label": true_label,
                "pred_label": pred_label,
                "confidence": confidence,
                "true_prob": true_prob,
                "is_correct": is_correct,
                "probs": probs.tolist()
            })

            if i % 50 == 0:
                print(f"{dataset_name}: {i}/{len(dataset)} | running acc: {np.mean([r['is_correct'] for r in results]):.3f}")

        except Exception as e:
            print(f"Skipped sample {i}: {e}")
            continue

    return results

In [11]:
wandb.init(project="Order_of_Operations", name="pipeline_b_eval")

mmlu_data = load_mmlu(300)
arc_data = load_arc(300)
tqa_data = load_truthfulqa(300)

results_mmlu = evaluate_pipeline(model, tokenizer, mmlu_data, format_mmlu, "MMLU", answer_token_ids)
results_arc = evaluate_pipeline(model, tokenizer, arc_data, format_arc, "ARC", answer_token_ids)
results_tqa = evaluate_pipeline(model, tokenizer, tqa_data, format_truthfulqa, "TruthfulQA", answer_token_ids)

all_results = results_mmlu + results_arc + results_tqa
print(f"\nTotal samples evaluated: {len(all_results)}")

total_flos,2126993950642176
train/entropy,1.30862
train/epoch,2
train/global_step,100
train/grad_norm,0.72656
train/learning_rate,0.0
train/loss,1.30081
train/mean_token_accuracy,0.66509
train/num_tokens,133556
train_loss,1.38149
+3,...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

multiple_choice/validation-00000-of-0000(…):   0%|          | 0.00/271k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]

MMLU: 0/300 | running acc: 0.000
MMLU: 50/300 | running acc: 0.216
MMLU: 100/300 | running acc: 0.337
MMLU: 150/300 | running acc: 0.397
MMLU: 200/300 | running acc: 0.373
MMLU: 250/300 | running acc: 0.382
ARC: 0/300 | running acc: 1.000
ARC: 50/300 | running acc: 0.490
ARC: 100/300 | running acc: 0.535
ARC: 150/300 | running acc: 0.517
ARC: 200/300 | running acc: 0.517
ARC: 250/300 | running acc: 0.522
TruthfulQA: 0/300 | running acc: 0.000
TruthfulQA: 50/300 | running acc: 0.235
TruthfulQA: 100/300 | running acc: 0.238
TruthfulQA: 150/300 | running acc: 0.252
TruthfulQA: 200/300 | running acc: 0.269
TruthfulQA: 250/300 | running acc: 0.271

Total samples evaluated: 900


In [12]:
!pip install netcal

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.3/236.3 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.2/291.2 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.8/174.8 kB 19.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Succ

In [13]:
from netcal.metrics import ECE
from sklearn.metrics import brier_score_loss
import torch.nn.functional as F

def compute_metrics(results, pipeline_name):
    confidences = np.array([r["confidence"] for r in results])
    true_probs = np.array([r["true_prob"] for r in results])
    is_correct = np.array([r["is_correct"] for r in results])

    # ECE
    ece = ECE(bins=15)
    ece_score = ece.measure(confidences, is_correct)

    # Brier Score (binary: was the top prediction correct?)
    brier = brier_score_loss(is_correct, confidences)

    # NLL (over true class probability)
    nll = -np.mean(np.log(true_probs + 1e-10))

    # Accuracy
    accuracy = np.mean(is_correct)

    # Confidence when correct vs incorrect
    conf_correct = np.mean(confidences[is_correct == 1]) if is_correct.sum() > 0 else 0
    conf_incorrect = np.mean(confidences[is_correct == 0]) if (1-is_correct).sum() > 0 else 0

    # Overconfidence rate: wrong but confidence > 0.7
    overconf_rate = np.mean((confidences > 0.7) & (is_correct == 0))

    # Underconfidence rate: right but confidence < 0.5
    underconf_rate = np.mean((confidences < 0.5) & (is_correct == 1))

    metrics = {
        "pipeline": pipeline_name,
        "accuracy": accuracy,
        "ECE": ece_score,
        "brier_score": brier,
        "NLL": nll,
        "conf_when_correct": conf_correct,
        "conf_when_incorrect": conf_incorrect,
        "overconfidence_rate": overconf_rate,
        "underconfidence_rate": underconf_rate,
    }

    for k, v in metrics.items():
        if k != "pipeline":
            print(f"  {k}: {v:.4f}")

    wandb.log({f"pipeline_b/{k}": v for k, v in metrics.items() if k != "pipeline"})

    return metrics

print("=== Pipeline B (QLoRA) ===")
metrics_b = compute_metrics(all_results, "Pipeline_B_QLoRA")

=== Pipeline B (QLoRA) ===
  accuracy: 0.3811
  ECE: 0.0383
  brier_score: 0.2226
  NLL: 1.3589
  conf_when_correct: 0.4487
  conf_when_incorrect: 0.3898
  overconfidence_rate: 0.0089
  underconfidence_rate: 0.2800


In [17]:
import pandas as pd
df = pd.DataFrame(all_results)
df.to_csv("/content/drive/MyDrive/pipeline_b_results.csv", index=False)

artifact = wandb.Artifact("pipeline_b_results", type="dataset")
artifact.add_file("/content/drive/MyDrive/pipeline_b_results.csv")
wandb.log_artifact(artifact)

# Also log the summary metrics directly
wandb.log({
    "pipeline_b/accuracy": metrics_b["accuracy"],
    "pipeline_b/ECE": metrics_b["ECE"],
    "pipeline_b/brier_score": metrics_b["brier_score"],
    "pipeline_b/NLL": metrics_b["NLL"],
    "pipeline_b/conf_when_correct": metrics_b["conf_when_correct"],
    "pipeline_b/conf_when_incorrect": metrics_b["conf_when_incorrect"],
    "pipeline_b/overconfidence_rate": metrics_b["overconfidence_rate"],
    "pipeline_b/underconfidence_rate": metrics_b["underconfidence_rate"],
})

wandb.finish()
print("Saved to Drive + W&B.")

pipeline_b/ECE,▁
pipeline_b/NLL,▁
pipeline_b/accuracy,▁
pipeline_b/brier_score,▁
pipeline_b/conf_when_correct,▁
pipeline_b/conf_when_incorrect,▁
pipeline_b/overconfidence_rate,▁
pipeline_b/underconfidence_rate,▁
pipeline_b/ECE,0.03834
pipeline_b/NLL,1.35888
pipeline_b/accuracy,0.38111


Saved to Drive + W&B.
